# 📊 Sonar Rock vs Mine — SVM (RBF): 208 vs 1,000 Rows

This notebook consolidates the supplied model notebooks into a reproducible **208-row vs 1,000-row comparison**.

### Comparison protocol
- 60 sonar measurements are used as features.
- Column 60 is the target (`R` = Rock, `M` = Mine).
- 90% training / 10% test split.
- `stratify=y` and `random_state=1`.
- 5-fold stratified cross-validation with `shuffle=True, random_state=1`.
- Interactive Plotly charts and an interactive KPI dashboard.

> **Data-quality note:** The 1,000-row dataset is an expanded/synthetic dataset. Its results should not be presented as equivalent to testing on 1,000 independently collected real sonar observations.

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RANDOM_STATE = 1
TEST_SIZE = 0.10
N_SPLITS = 5

DATASETS = {
    "Original 208 rows": "Copy of sonar data.csv",
    "Expanded 1000 rows": "Sonar_Rock_vs_Mine_1000.csv"
}

## 2. Load and validate both datasets

In [ ]:
def find_file(filename):
    candidates = [
        Path(filename),
        Path("/mnt/data") / filename,
        Path("/content") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find '{filename}'. Place both CSV files in the notebook working directory."
    )

def load_dataset(filename, expected_rows):
    path = find_file(filename)
    df = pd.read_csv(path, header=None)

    if df.shape != (expected_rows, 61):
        raise ValueError(
            f"{filename}: expected shape ({expected_rows}, 61), found {df.shape}"
        )

    if not set(df.iloc[:,60].astype(str).unique()).issubset({"R","M"}):
        raise ValueError(f"{filename}: target column must contain only R/M.")

    return df

df_208 = load_dataset(DATASETS["Original 208 rows"], 208)
df_1000 = load_dataset(DATASETS["Expanded 1000 rows"], 1000)

print("Original dataset:", df_208.shape)
print("Expanded dataset:", df_1000.shape)

## 3. Dataset overview and class distribution

In [ ]:
overview = pd.DataFrame({
    "Dataset": ["Original 208", "Expanded 1000"],
    "Rows": [len(df_208), len(df_1000)],
    "Features": [60, 60],
    "Rock": [(df_208[60]=="R").sum(), (df_1000[60]=="R").sum()],
    "Mine": [(df_208[60]=="M").sum(), (df_1000[60]=="M").sum()]
})
display(overview)

In [ ]:
class_df = pd.DataFrame({
    "Dataset": ["Original 208", "Original 208", "Expanded 1000", "Expanded 1000"],
    "Class": ["Rock", "Mine", "Rock", "Mine"],
    "Count": [
        (df_208[60]=="R").sum(), (df_208[60]=="M").sum(),
        (df_1000[60]=="R").sum(), (df_1000[60]=="M").sum()
    ]
})

fig = px.bar(
    class_df, x="Dataset", y="Count", color="Class", barmode="group",
    text="Count", title="Interactive Class Distribution"
)
fig.update_layout(template="plotly_white")
fig.show()

## 4. Model definition

In [ ]:
def build_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            probability=True,
            random_state=1
        ))
    ])

## 5. Train, test and 5-fold cross-validate both datasets

In [ ]:
def run_experiment(df, name):
    X = df.iloc[:, :60]
    y = df.iloc[:, 60]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )

    model = build_model()
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )
    cv_scores = cross_val_score(
        build_model(), X, y, cv=cv, scoring="accuracy"
    )

    return {
        "name": name,
        "rows": len(df),
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "cv_scores": cv_scores,
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "cm": confusion_matrix(y_test, test_pred, labels=["R","M"]),
        "report": classification_report(
            y_test, test_pred,
            labels=["R","M"],
            target_names=["Rock","Mine"],
            output_dict=True
        ),
        "model": model,
        "test_pred": test_pred,
        "y_test": y_test
    }

results = {
    "208": run_experiment(df_208, "Original 208 rows"),
    "1000": run_experiment(df_1000, "Expanded 1000 rows")
}

## 6. Accuracy summary

In [ ]:
summary = pd.DataFrame([
    {
        "Dataset": results["208"]["name"],
        "Rows": results["208"]["rows"],
        "Train Accuracy": results["208"]["train_accuracy"],
        "Test Accuracy": results["208"]["test_accuracy"],
        "CV Mean": results["208"]["cv_mean"],
        "CV Std": results["208"]["cv_std"]
    },
    {
        "Dataset": results["1000"]["name"],
        "Rows": results["1000"]["rows"],
        "Train Accuracy": results["1000"]["train_accuracy"],
        "Test Accuracy": results["1000"]["test_accuracy"],
        "CV Mean": results["1000"]["cv_mean"],
        "CV Std": results["1000"]["cv_std"]
    }
])

display(
    summary.style.format({
        "Train Accuracy":"{:.2%}",
        "Test Accuracy":"{:.2%}",
        "CV Mean":"{:.2%}",
        "CV Std":"{:.2%}"
    })
)

## 7. Detailed test-set classification reports

In [ ]:
for key in ["208", "1000"]:
    r = results[key]
    print("\n" + "="*72)
    print(r["name"])
    print(classification_report(
        r["y_test"], r["test_pred"],
        labels=["R","M"],
        target_names=["Rock","Mine"],
        digits=4
    ))

## 8. Interactive accuracy comparison

In [ ]:
accuracy_df = pd.DataFrame([
    {"Dataset":"Original 208","Metric":"Training","Accuracy":results["208"]["train_accuracy"]},
    {"Dataset":"Original 208","Metric":"Test","Accuracy":results["208"]["test_accuracy"]},
    {"Dataset":"Original 208","Metric":"5-Fold CV Mean","Accuracy":results["208"]["cv_mean"]},
    {"Dataset":"Expanded 1000","Metric":"Training","Accuracy":results["1000"]["train_accuracy"]},
    {"Dataset":"Expanded 1000","Metric":"Test","Accuracy":results["1000"]["test_accuracy"]},
    {"Dataset":"Expanded 1000","Metric":"5-Fold CV Mean","Accuracy":results["1000"]["cv_mean"]},
])

fig = px.bar(
    accuracy_df, x="Metric", y="Accuracy", color="Dataset",
    barmode="group",
    text=accuracy_df["Accuracy"].map(lambda x: f"{x:.1%}"),
    title="SVM (RBF): Interactive Accuracy Comparison"
)
fig.update_yaxes(range=[0,1], tickformat=".0%")
fig.update_layout(template="plotly_white")
fig.show()

## 9. Interactive 5-fold cross-validation chart

In [ ]:
cv_rows = []
for key in ["208","1000"]:
    for fold, score in enumerate(results[key]["cv_scores"], start=1):
        cv_rows.append({
            "Dataset": results[key]["name"],
            "Fold": fold,
            "Accuracy": float(score)
        })

cv_df = pd.DataFrame(cv_rows)

fig = px.line(
    cv_df, x="Fold", y="Accuracy", color="Dataset",
    markers=True, text=cv_df["Accuracy"].map(lambda x: f"{x:.1%}"),
    title="Interactive 5-Fold Cross-Validation Performance"
)
fig.update_yaxes(range=[0.6,1.0], tickformat=".0%")
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()

## 10. Interactive confusion-matrix comparison

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Original 208 rows", "Expanded 1000 rows")
)

for col, key in enumerate(["208","1000"], start=1):
    cm = results[key]["cm"]
    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=["Predicted Rock","Predicted Mine"],
            y=["Actual Rock","Actual Mine"],
            text=cm,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=(col==2),
            hovertemplate="Actual: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>"
        ),
        row=1, col=col
    )

fig.update_layout(
    title="Interactive Confusion Matrices",
    height=500,
    template="plotly_white"
)
fig.show()

## 11. Interactive precision / recall / F1 analysis

In [ ]:
metric_rows = []
for key in ["208","1000"]:
    rep = results[key]["report"]
    for cls in ["Rock","Mine"]:
        metric_rows += [
            {"Dataset":results[key]["name"],"Class":cls,"Metric":"Precision","Score":rep[cls]["precision"]},
            {"Dataset":results[key]["name"],"Class":cls,"Metric":"Recall","Score":rep[cls]["recall"]},
            {"Dataset":results[key]["name"],"Class":cls,"Metric":"F1-score","Score":rep[cls]["f1-score"]}
        ]

metric_df = pd.DataFrame(metric_rows)

fig = px.bar(
    metric_df, x="Metric", y="Score", color="Dataset",
    facet_col="Class", barmode="group",
    text=metric_df["Score"].map(lambda x: f"{x:.1%}"),
    title="Interactive Classification Metrics"
)
fig.update_yaxes(range=[0,1], tickformat=".0%")
fig.update_layout(template="plotly_white")
fig.show()

## 12. Interactive KPI dashboard

In [ ]:
dashboard = make_subplots(
    rows=2, cols=2,
    specs=[[{"type":"indicator"},{"type":"indicator"}],
           [{"type":"indicator"},{"type":"indicator"}]],
    vertical_spacing=0.18
)

kpis = [
    (results["208"]["test_accuracy"], "208-row Test Accuracy"),
    (results["1000"]["test_accuracy"], "1000-row Test Accuracy"),
    (results["208"]["cv_mean"], "208-row CV Mean"),
    (results["1000"]["cv_mean"], "1000-row CV Mean")
]
positions = [(1,1),(1,2),(2,1),(2,2)]

for (value, title), (row,col) in zip(kpis, positions):
    dashboard.add_trace(
        go.Indicator(
            mode="number+gauge",
            value=value*100,
            number={"suffix":"%","valueformat":".2f"},
            title={"text":title},
            gauge={"axis":{"range":[0,100]}}
        ),
        row=row, col=col
    )

dashboard.update_layout(
    title="SVM (RBF) — 208 vs 1000 Interactive Dashboard",
    height=700,
    template="plotly_white"
)
dashboard.show()

## 13. Quantitative change from 208 → 1,000 rows

In [ ]:
test_delta_pp = (results["1000"]["test_accuracy"] - results["208"]["test_accuracy"]) * 100
cv_delta_pp = (results["1000"]["cv_mean"] - results["208"]["cv_mean"]) * 100
train_delta_pp = (results["1000"]["train_accuracy"] - results["208"]["train_accuracy"]) * 100

print(f"Test accuracy change     : {test_delta_pp:+.2f} percentage points")
print(f"CV mean accuracy change  : {cv_delta_pp:+.2f} percentage points")
print(f"Training accuracy change : {train_delta_pp:+.2f} percentage points")

## 12. Detailed interpretation — SVM (RBF)

### Original 208-row dataset
- Training accuracy: **100.00%**
- Test accuracy: **95.24%**
- Mean 5-fold CV accuracy: **87.54% ± 4.02%**
- Confusion matrix: `[[9, 1], [0, 11]]`

The held-out test split is very strong, including **100% recall for Mine** on that split. However, the 21-sample test set is small, and the lower cross-validation mean shows that the 95.24% test result is optimistic relative to performance across folds.

### Expanded 1,000-row dataset
- Training accuracy: **100.00%**
- Test accuracy: **99.00%**
- Mean 5-fold CV accuracy: **99.90% ± 0.20%**
- Confusion matrix: `[[47, 0], [1, 52]]`

Compared with 208 rows, the expanded experiment improves:
- Test accuracy by **3.81 percentage points**.
- Mean CV accuracy by **12.36 percentage points**.

### Interpretation
SVM is the strongest model in these experiments. The extremely high 99.90% cross-validation score on the synthetic 1,000-row dataset should be treated cautiously because synthetic observations can be more similar to one another than independent real measurements.

## 13. Predictive-system check using the same 60-feature sample

In [ ]:
sample_input = np.asarray([0.0442, 0.1075, 0.0549, 0.0278, 0.0753, 0.1517, 0.1777, 0.2277, 0.2668, 0.1826, 0.1874, 0.2698, 0.3506, 0.3143, 0.308, 0.4399, 0.4245, 0.1777, 0.3984, 0.391, 0.5339, 0.546, 0.5167, 0.6315, 0.5091, 0.4786, 0.6901, 0.8766, 0.8916, 0.9025, 0.8489, 0.6961, 0.3579, 0.4006, 0.4978, 0.3154, 0.2648, 0.3034, 0.2202, 0.1854, 0.6177, 0.6973, 0.5677, 0.4301, 0.5366, 0.5901, 0.3928, 0.2275, 0.1366, 0.0649, 0.0699, 0.0394, 0.0182, 0.0235, 0.0126, 0.0155, 0.0074, 0.019, 0.0091, 0.0084], dtype=float).reshape(1,-1)

if sample_input.shape[1] != 60:
    raise ValueError("The sample input must contain exactly 60 sonar measurements.")

for key in ["208","1000"]:
    model = results[key]["model"]
    prediction = model.predict(sample_input)[0]
    probabilities = model.predict_proba(sample_input)[0]
    prob_map = dict(zip(model.classes_, probabilities))

    label = "Mine" if prediction == "M" else "Rock"
    print(f"{results[key]['name']} → {label}")
    print("  Class probabilities:",
          {cls: round(float(p),4) for cls,p in prob_map.items()})

## 14. Final conclusion

The comparison should use both the **held-out test accuracy** and the **5-fold cross-validation mean**.

For the expanded dataset, the larger sample produces higher scores. However, because it is synthetic/expanded rather than independently collected, the higher performance should be treated as an experimental result rather than a guaranteed real-world improvement.

Open this notebook in **Google Colab or Jupyter Notebook/Lab** to use the full Plotly interactivity.